# Implement ChebNet and classify papers in the Cora dataset

In this notebook, you will implement ChebNet, which is arguably one of the most influential graph neural networks of the convolutional spectral methods for graphs. You will:

1. Load the Cora citation network
2. Build graph Laplacians and the scaled Laplacian for ChebNet
3. Implement Chebyshev polynomials and a ChebNet layer
4. Train a simple single-layer ChebNet for semi-supervised node classification on Cora

All tasks are either marked with **TODO** (in markdown) or `# TODO` (in code).

A useful resource to learn about graph neural networks and especially ChebNet is [this lecture](https://youtu.be/4la2YC7vg8Q?t=4416) from the University of Guelph, Canada.

## Setup and utilities
Helper functions. **Read further before starting the TODOs in the code block below**.

In [ ]:

import os
import tarfile
import urllib.request as request

import numpy as np

np.set_printoptions(precision=4, suppress=True)

def degree_matrix(A):
    d = A.sum(axis=1)
    return np.diag(d)

def laplacian(A, normalized=False):
    ### TODO: Either compute the regular Laplacian or the normalized version based on the flag.
    ### Regular: D - A
    ### Normalized: I - D^(-1/2)AD^(-1/2)
    pass

def scaled_laplacian(L):
    ### TODO: Scale the Laplacian L to the range of -1,1.
    pass
    
def get_largest_eigenvalue(mat):
    return float(np.linalg.eigvalsh(mat).max())


## 1. Load the Cora dataset
The Cora dataset represents academic publications and citations, consisting of 2708 nodes (papers) and 5429 edges (citations). Each node has a binary word vector of size 1433 where each value represents the absence/presence of a specific word. The publications are assigned to one of seven classes, representing the field of the publication. In this exercise, we will use the dataset to predict the field of papers.

![Visualization of the Cora dataset](https://graphsandnetworks.com/wp-content/uploads/2019/09/CoraBalloons.png)

In [ ]:
URL = "https://linqs-data.soe.ucsc.edu/public/lbc/cora.tgz"
DATA_DIR = "./"
TGZ_PATH = os.path.join(DATA_DIR, "cora.tgz")

os.makedirs(DATA_DIR, exist_ok=True)

### Download
if not os.path.exists(TGZ_PATH):
    print("Downloading Cora dataset...")
    request.urlretrieve(URL, TGZ_PATH)
else:
    print("File already exists:", TGZ_PATH)

### Extract
with tarfile.open(TGZ_PATH, "r:gz") as tar:
    tar.extractall(path=DATA_DIR)
print("Extracted to", DATA_DIR)

### Load cora.content, which contains nodes, features and labels
content_path = os.path.join(DATA_DIR, "cora", "cora.content")
ids = []
features = []
labels_raw = []
with open(content_path, "r") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) < 3:
            continue
        ids.append(parts[0])
        features.append([float(x) for x in parts[1:-1]])
        labels_raw.append(parts[-1])

ids = np.array(ids)
X = np.array(features, dtype=float) # matrix of dimensions (samples,features)

### Label encoding
classes = sorted(set(labels_raw))
label_map = {c: i for i, c in enumerate(classes)}
y = np.array([label_map[c] for c in labels_raw], dtype=int)

### Load cora.cites, which contains the edges, and store it in an adjacency matrix
n = len(ids)
# For each id (node identifiers given by the dataset), get the index (idx) 
id_to_idx = {pid: i for i, pid in enumerate(ids)}
A = np.zeros((n, n), dtype=float)

cites_path = os.path.join(DATA_DIR, "cora", "cora.cites")
with open(cites_path, "r") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) != 2:
            continue
        src, dst = parts
        if src in id_to_idx and dst in id_to_idx:
            i, j = id_to_idx[src], id_to_idx[dst]
            A[i, j] = 1.0
            A[j, i] = 1.0  # undirected

print("Nodes:", n, "Features:", X.shape[1], "Classes:", len(classes))
print("Adjacency shape:", A.shape)


## 1. Laplacian and scaled Laplacian
The Laplacian $L=D-A$ contains both the adjacency structure as well as the degree of nodes, essentially describing the whole graph. The normalization $L = I - D^{-1/2} A D^{-1/2}$ has a number of favorable properties mostly relevant for spectral graph theory; for the case of graph neural networks, the normalization can mostly be seen as reducing the impact of high-degree nodes relative to other nodes.

**TODO**: Compute the normalized Laplacian, then use it to compute the scaled Laplacian $\tilde L = (2 / \lambda_{max})L - I$. Note: Getting the largest Eigenvalue of $L$, $\lambda_{max}$, requires matrix inversion, which is generally of complexity $\mathcal{O}(n^3)$ and can be approximated for large graphs in practice.

In [ ]:
### TODO: Complete the code in the first code cell so that this block works.
L = laplacian(A, normalized=True)
L_tilde = scaled_laplacian(L)


## 2. Chebyshev polynomials on graphs
Chebyshev polynomials are used for approximations where uniform accuracy over the whole domain [-1,1] is relevant (compared to Taylor series, which are locally accurate).

**TODO**: Implement Chebyshev polynomials of a matrix operator (slightly different than for scalar values): 
$$T_0(\tilde L)X=X\\T_1(\tilde L)X=\tilde L X\\T_k(\tilde L)X=2 \tilde L T_{k-1}(\tilde L)X - T_{k-2}(\tilde L)X$$


In [ ]:

def chebyshev_polynomials(L_tilde, X, K):
    ### TODO: Fill this function. It returns a list of length K representing the approxmiations for i=0..K.
    pass

for K in [0,1,2,3]:
    Ts = chebyshev_polynomials(L_tilde, X, K)
    print(f"K={K}", [t.shape for t in Ts]) # Should be the same shape for every K, but +1 entry.


K=0 [(2708, 1433)]
K=1 [(2708, 1433), (2708, 1433)]
K=2 [(2708, 1433), (2708, 1433), (2708, 1433)]
K=3 [(2708, 1433), (2708, 1433), (2708, 1433), (2708, 1433)]


## 3. ChebNet layer
The idea of ChebNet is that since convolution in the graph domain corresponds to the element-wise product in the spectral domain (Eigenvalue decomposition; $h*g = U\hat GU^Th$), the spectral filters $\hat g_v$ for nodes $v$ that make up $\hat G$ can be approximated. These approximations are parameterized with learned coefficients $\theta$.


**TODO**: Fill the blanks in the following sentences: ChebNet uses Chebyshev polynomials to approximate the **___** of the scaled Laplacian $\tilde L$. It does so to avoid computationally costly **___  ___**.

**TODO**: Implement the ChebNet layer $Z = \sum_{i=0}^k T_i(\tilde L) X \theta_i$, with coefficients $\theta_i \in \mathbb{R}^{F_{in} \times F_{out}}$.


In [ ]:
def chebnet_forward_pass(TX, thetas):
    ### TODO: Fill this function. It builds the actual approximation of the eigenvalues by summing
    ### up the products of the Chebyshev polynomial with the parameters theta over all choices of K. 
    pass

### For demo: Map the input feature dimension to the output feature dimension of 10  
F_in, F_out = X.shape[1], 10
K = 2
thetas_init = [np.random.randn(F_in, F_out)*0.1 for _ in range(K+1)]

### Compute polynomials once, then perform a forward pass
TX = chebyshev_polynomials(L_tilde, X, K)
Z_demo = chebnet_forward_pass(TX, thetas_init)
print("Z_demo shape:", Z_demo.shape) # 


Z_demo shape: (2708, 10)


## 4. Semi-supervised split
Use a small labeled set per class for training, hold-out validation, and the rest for test. As the nodes that are neither in training, validation, nor test set still influence the training through the parameter K (order of polynomials & receptive field), this split of the dataset can be seen as semi-supervised. 

In [6]:
def canonical_cora_split(y, train_per_class, val_size, test_size, seed=42):
    rng = np.random.RandomState(seed)
    n = len(y)
    classes = np.unique(y)

    # 20 labeled nodes per class
    train_idx = []
    for c in classes:
        idx_c = np.where(y == c)[0]
        rng.shuffle(idx_c)
        take = min(train_per_class, len(idx_c))
        train_idx.append(idx_c[:take])
    train_idx = np.concatenate(train_idx)

    # From the remaining, sample 500 val and 1000 test
    remaining = np.setdiff1d(np.arange(n), train_idx, assume_unique=False)
    rng.shuffle(remaining)

    val_take = min(val_size, len(remaining))
    val_idx = remaining[:val_take]

    remaining2 = remaining[val_take:]
    test_take = min(test_size, len(remaining2))
    test_idx = remaining2[:test_take]

    return train_idx, val_idx, test_idx

### Cora has n=2708 and 7 classes
train_idx, val_idx, test_idx = canonical_cora_split(y, 20, 500, 1000)
print(len(train_idx), len(val_idx), len(test_idx))

140 500 1000


## 5. Train single-layer ChebNet with manual gradients
For the forward pass, compute $Z=\sum_k T_k(\tilde L)X\theta_k$. Then compute the gradients on $Z$

Forward: Z = sum_k T_k X Theta_k; Softmax; Cross-entropy on labeled nodes only. Backprop only to Theta_k. Precompute T_k X.


In [ ]:

def one_hot(y, C):
    Y = np.zeros((len(y), C), dtype=float)
    Y[np.arange(len(y)), y] = 1.0
    return Y

def softmax(logits):
    logits = logits - logits.max(axis=1, keepdims=True)
    e = np.exp(logits)
    return e / (e.sum(axis=1, keepdims=True) + 1e-12)

def accuracy(logits, y_true):
    return float((logits.argmax(axis=1) == y_true).mean())

def compute_grads(TX, Thetas, idx, y_true):
    logits = chebnet_forward_pass(TX, Thetas)
    P = softmax(logits)
    Y = one_hot(y_true, P.shape[1])
    G = np.zeros_like(P)
    G[idx] = (P[idx] - Y)
    grads = [TX[k].T @ G for k in range(len(Thetas))]
    loss = -np.mean(np.log(P[idx, y_true] + 1e-12))
    return grads, loss, logits

def step_adam(params, grads, m, v, t, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=5e-4):
    t += 1
    for i in range(len(params)):
        g = grads[i] + weight_decay*params[i]
        m[i] = beta1*m[i] + (1-beta1)*g
        v[i] = beta2*v[i] + (1-beta2)*(g*g)
        mhat = m[i] / (1-beta1**t)
        vhat = v[i] / (1-beta2**t)
        params[i] -= lr * mhat / (np.sqrt(vhat) + eps)
    return params, m, v, t

### Hyperparameters, feel free to play around with them
K = 2
epochs = 200
patience = 20

TX = chebyshev_polynomials(L_tilde, X, K)

Fin = X.shape[1]; Fout = int(y.max()+1)
thetas = [np.random.randn(Fin, Fout)*0.01 for _ in range(K+1)]
momentum = [np.zeros_like(t) for t in thetas]
velocity = [np.zeros_like(t) for t in thetas]

### Initialization
t_adam = 0
best_val = -1e9
best_params = None
patience_left = patience

### Training loop
for epoch in range(1, epochs+1):
    ### TODO: Implement training loop: Compute gradients on the forward pass of the polynomials,
    ### take a step with adam, compute validation loss and then accuracy for both train and val.

    ...

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:03d} | train {train_loss:.4f}/{train_acc:.3f} | val {val_loss:.4f}/{val_acc:.3f}")
    
    if val_acc > best_val:
        best_val, best_params, patience_left = val_acc, [p.copy() for p in thetas], patience
    else:
        patience_left -= 1
        if patience_left == 0:
            break

if best_params is not None:
    thetas = best_params

test_logits = chebnet_forward_pass(TX, thetas)
test_acc = accuracy(test_logits[test_idx], y[test_idx])
print("Test accuracy:", round(test_acc, 4))


## 6. The influence of K
In addition to reducing the approximation error by allowing bigger polynomials, the parameter $K$ determines from how many hops (jumps from one node to another) away information is aggregated (receptive field). It is analogous to the number of message passing steps in spatial GNNs such as GraphSAGE.

**TODO**: Train for bigger K, at least up to 6. Does the test accuracy improve for each increased K? Do you observe oversmoothing? Reason why this might be the case.


In [ ]:

def train_chebnet_K(K, lr=0.02, epochs=150):
    Lt = scaled_laplacian(laplacian(A))
    TX = chebyshev_polynomials(Lt, X, K)
    Fin = X.shape[1]
    Fout = int(y.max()+1)
    Thetas = [np.random.randn(Fin, Fout) * 0.01 for _ in range(K+1)]
    momentum = [np.zeros_like(t) for t in Thetas]
    velocity = [np.zeros_like(t) for t in Thetas]
    t_adam = 0; best_val = -1e9; best = None; patience_left = 15
    for epoch in range(1, epochs+1):
        ### TODO: Copy the training loop code that you wrote in the above cell.

        if val_acc > best_val:
            best_val, best, patience_left = val_acc, [t.copy() for t in Thetas], 15
        else:
            patience_left -= 1
            if patience_left == 0:
                break
    if best is not None:
        Thetas = best
    test_acc = accuracy(chebnet_forward_pass(TX, Thetas)[test_idx], y[test_idx])
    return best_val, test_acc

for K in range(20):
    val_acc, test_acc = train_chebnet_K(K)
    print(f"K={K} -> val={val_acc:.3f}, test={test_acc:.3f}")


## Questions
**TODO**: Answer the following questions.
1. Into what range is $\tilde L$ scaled with $\lambda_{max}$? And why is this done in ChebNet?

2. How does the parameter $K$ relate to which neighbors a node aggregates information from? What are the upsides and downsides of choosing large or small values of K?

3. Conceptually, how can the above training of ChebNet be extended to two or more ChebNet layers? What would be the input to the second layer and what would be its output?

4. Investigate: What changes does GCN (Kipf and Welling, 2017) propose on top of ChebNet? Why?


# Further reading
If you are interested, some deep learning libraries for graphs are [Graph Nets](https://github.com/google-deepmind/graph_nets) built on Tensorflow and [PyTorch Geometric](https://pytorch-geometric.readthedocs.io/en/latest/). Maybe you can come up with a model with better accuracy on the Cora dataset? (Definitely optional!)